# AI Worker Burnout Prediction

A corrected, reproducible regression experiment by Dylan Wilson.

**Question:** how well do workplace and AI-related variables predict the recorded `burnout_score` within this dataset?

This is exploratory same-record prediction, not a causal study or a validated prospective burnout system. Data provenance, scoring definitions, and licensing remain unverified; see [dataset notes](data/README.md).

## 1. Load the shared implementation

The CLI and notebook use the same functions. No Colab upload or out-of-order hidden variables are required. Run this notebook with its working directory at the repository root.

In [1]:
from pathlib import Path
import importlib.util
import json
import numpy as np
from IPython.display import display, Image

ROOT = Path.cwd()
if not (ROOT / 'code.py').is_file():
    raise FileNotFoundError('Start Jupyter from the repository root containing code.py.')
spec = importlib.util.spec_from_file_location('burnout_project', ROOT / 'code.py')
burnout = importlib.util.module_from_spec(spec)
spec.loader.exec_module(burnout)
DATA_PATH = ROOT / 'ai_worker_burnout_attrition_2026.csv'
OUTPUT_DIR = ROOT / 'reports' / 'scratch' / 'notebook'


## 2. Inspect deterministic cleaning

Trim category whitespace and convert blank strings to missing values. Remove exact duplicate records and invalid targets; conflicting employee IDs require review. Missing predictors are left for fold-specific imputers.

**Retain extreme scores:** IQR outliers are not automatically erroneous. Filtering by the target before evaluation would change which workers the model is evaluated on.

In [2]:
df, audit = burnout.load_data(DATA_PATH)
print(json.dumps(audit, indent=2))
display(df.head())


{
  "original_rows": 1500,
  "original_columns": 21,
  "exact_duplicates_removed": 0,
  "invalid_targets_removed": 0,
  "retained_rows": 1500,
  "missing_values": {
    "employee_id": 0,
    "job_role": 0,
    "years_experience": 0,
    "education_level": 0,
    "country": 0,
    "industry": 0,
    "company_size": 0,
    "remote_work_type": 0,
    "team_size": 0,
    "salary_usd_k": 0,
    "primary_ai_tool": 0,
    "ai_tools_used_per_day": 0,
    "hours_with_ai_assistance_daily": 0,
    "ai_replaces_my_tasks_pct": 0,
    "ai_adoption_stage": 0,
    "weekly_ai_upskilling_hrs": 0,
    "productivity_score": 0,
    "burnout_score": 0,
    "job_satisfaction_1_5": 0,
    "fear_of_ai_replacement": 0,
    "attrition_risk": 0
  },
  "outlier_policy": "Retain extremes; no IQR-based row removal."
}


,employee_id,job_role,years_experience,education_level,country,industry,company_size,remote_work_type,team_size,salary_usd_k,primary_ai_tool,ai_tools_used_per_day,hours_with_ai_assistance_daily,ai_replaces_my_tasks_pct,ai_adoption_stage,weekly_ai_upskilling_hrs,productivity_score,burnout_score,job_satisfaction_1_5,fear_of_ai_replacement,attrition_risk
0,EMP0001,Data Analyst,7,PhD,India,Healthtech,Large (1000-5000),Fully Remote,8,184,GitHub Copilot,1,0.6,37,Experimenting,4.9,59,58,3.0,Low,Medium
1,EMP0002,DevOps Engineer,3,Self-taught,Canada,Fintech,Large (1000-5000),Hybrid,42,51,ChatGPT,3,0.6,78,Experimenting,6.6,64,47,3.6,Medium,Medium
2,EMP0003,Prompt Engineer,3,PhD,India,Automotive,Startup (<50),Hybrid,37,185,Midjourney,4,4.0,23,Optimizing,5.8,50,37,4.6,High,Medium
3,EMP0004,Backend Engineer,18,Master,Germany,Media,Mid (200-1000),Fully Remote,8,131,Perplexity,5,4.2,75,Integrating,1.4,77,67,3.0,Low,Medium
4,EMP0005,AI Researcher,5,Bachelor,India,E-commerce,Small (50-200),Fully Remote,21,99,Notion AI,7,0.7,61,Experimenting,1.6,58,54,3.6,Medium,Medium


## 3. Define features and split before fitting preprocessing

Exclude `employee_id`, `burnout_score`, and `attrition_risk`. The last field may encode an outcome related to burnout; its construction is unknown, so it is excluded conservatively. This is not proof of leakage.

The retained `job_satisfaction_1_5` and `productivity_score` also have unknown measurement timing and may depend on burnout. Results therefore describe concurrent records, not future burnout forecasts.

In [3]:
X, y = burnout.make_features(df)
X_train, X_test, y_train, y_test = burnout.train_test_split(
    X, y, test_size=0.2, random_state=burnout.SEED
)
assert set(X_train.index).isdisjoint(X_test.index)
assert not set(burnout.EXCLUDED).intersection(X.columns)
print(f'Training: {len(X_train)} | Test: {len(X_test)} | Predictors: {X.shape[1]}')
display(X_train.describe())


Training: 1200 | Test: 300 | Predictors: 18


,years_experience,team_size,salary_usd_k,ai_tools_used_per_day,hours_with_ai_assistance_daily,ai_replaces_my_tasks_pct,weekly_ai_upskilling_hrs,productivity_score,job_satisfaction_1_5
count,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000
mean,10.048333,24.882500,140.214167,4.019167,4.234000,41.153333,5.008167,57.351667,3.344500
std,5.480666,13.727918,59.635842,1.978736,2.155501,21.617530,2.901591,10.648434,0.635694
min,1.000000,2.000000,40.000000,1.000000,0.500000,5.000000,0.000000,28.000000,1.300000
25%,5.000000,13.000000,91.000000,2.000000,2.400000,23.000000,2.500000,50.000000,2.900000
50%,10.000000,25.000000,137.500000,4.000000,4.200000,40.000000,5.000000,57.000000,3.300000
75%,15.000000,37.000000,189.000000,6.000000,6.100000,60.000000,7.500000,65.000000,3.800000
max,19.000000,49.000000,248.000000,7.000000,8.000000,79.000000,10.000000,95.000000,5.000000


## 4. Inspect the preprocessing and model configurations

Each pipeline fits median imputation and scaling for numerical features, and mode imputation plus one-hot encoding for categorical features. Categories are never represented by arbitrary ordered labels. One category is dropped per field to avoid redundant columns in linear regression; unknown categories are accepted at inference time.

A shared five-fold shuffled KFold (seed 42) evaluates all metrics on **training records only**. Preprocessing is refit within each fold. Select by the lowest mean CV RMSE, then fit on the full training split. The final test data is not used for model selection.

In [4]:
models = burnout.make_models(X_train)
print('Models:', ', '.join(models))
display(models['Linear Regression'])


Models: Mean Baseline, Linear Regression, Decision Tree, Random Forest


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tra

## 5. Run the experiment from raw data

`run_experiment` performs cleaning, splitting, training-only CV, selection, final fits, and test evaluation in a fixed order. It also exports full metrics, dataset/split fingerprints, diagnostic figures, and original-feature permutation importance.

Outputs from this notebook go to `reports/scratch/notebook/`; the checked-in CLI results are preserved.

In [5]:
experiment = burnout.run_experiment(DATA_PATH, OUTPUT_DIR)
comparison = experiment['comparison']
selected = experiment['metadata']['selected_model']
display(comparison)
print('Selected by training CV:', selected)


            model  cv_rmse_mean  cv_r2_mean  test_mae  test_rmse   test_r2
Linear Regression      7.361303    0.585216  5.418714   6.665334  0.655234
    Random Forest      7.463010    0.573372  5.608869   6.968065  0.623205
    Decision Tree      9.231061    0.346522  7.622068   9.169699  0.347485
    Mean Baseline     11.458059   -0.003996  9.263178  11.351889 -0.000038

Selected by training CV: Linear Regression
Selected by training CV: Linear Regression


,model,cv_mae_mean,cv_mae_std,cv_rmse_mean,cv_rmse_std,cv_r2_mean,cv_r2_std,cv_fit_seconds_mean,train_mae,train_rmse,train_r2,test_mae,test_rmse,test_r2,test_predict_seconds
0,Linear Regression,5.854218,0.280488,7.361303,0.337058,0.585216,0.027614,0.009634,5.570683,7.013685,0.625072,5.418714,6.665334,0.655234,0.003353
1,Random Forest,5.887896,0.329583,7.463010,0.373970,0.573372,0.035821,0.373488,3.683698,4.811783,0.823531,5.608869,6.968065,0.623205,0.008612
2,Decision Tree,7.377116,0.267441,9.231061,0.310536,0.346522,0.054081,0.014608,3.786341,4.817647,0.823101,7.622068,9.169699,0.347485,0.003397
3,Mean Baseline,9.208665,0.282474,11.458059,0.371080,-0.003996,0.004952,0.008535,9.200067,11.454393,0.000000,9.263178,11.351889,-0.000038,0.003346


## 6. Compare against the baseline

MAE is the average absolute error in burnout-score units. RMSE gives larger errors more weight. R² compares the squared prediction error with a constant prediction of the evaluated target mean; it is not classification accuracy.

Training versus test gaps help identify overfitting. CV variability below is fold-to-fold spread, not a confidence interval. Fixed settings were retained rather than tuned to the test set.

In [6]:
columns = ['model', 'train_r2', 'cv_rmse_mean', 'cv_rmse_std', 'cv_r2_mean', 'test_mae', 'test_rmse', 'test_r2']
display(comparison[columns])
display(Image(filename=str(OUTPUT_DIR / 'figures' / 'model_comparison.png')))


,model,train_r2,cv_rmse_mean,cv_rmse_std,cv_r2_mean,test_mae,test_rmse,test_r2
0,Linear Regression,0.625072,7.361303,0.337058,0.585216,5.418714,6.665334,0.655234
1,Random Forest,0.823531,7.463010,0.373970,0.573372,5.608869,6.968065,0.623205
2,Decision Tree,0.823101,9.231061,0.310536,0.346522,7.622068,9.169699,0.347485
3,Mean Baseline,0.000000,11.458059,0.371080,-0.003996,9.263178,11.351889,-0.000038


<IPython.core.display.Image object>

## 7. Inspect errors and feature importance

The dashed diagonal represents perfect agreement, not a separate set of actual observations. Residuals are actual minus predicted; structure in residuals can indicate unmodeled patterns but does not by itself diagnose a specific cause.

Permutation importance shuffles each original feature 10 times and measures the drop in test R² for the selected model. It is descriptive and not used to revise the model after looking at test results. Correlation and predictive importance do not establish causality.

![Predictions](reports/figures/predictions.png)

![Residuals](reports/figures/residuals.png)

![Permutation importance](reports/figures/feature_importance.png)


![Predictions](reports/figures/predictions.png)

![Residuals](reports/figures/residuals.png)

![Permutation importance](reports/figures/feature_importance.png)


In [7]:
for filename in ['predictions.png', 'residuals.png', 'feature_importance.png']:
    display(Image(filename=str(OUTPUT_DIR / 'figures' / filename)))
display(experiment['importance'].head(10))


<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

,feature,importance_mean,importance_std
12,ai_replaces_my_tasks_pct,0.435085,0.038948
16,job_satisfaction_1_5,0.292523,0.047933
11,hours_with_ai_assistance_daily,0.133668,0.029031
4,industry,0.005125,0.004336
0,job_role,0.004414,0.002188
3,country,0.004333,0.004077
6,remote_work_type,0.001369,0.001084
13,ai_adoption_stage,0.000137,0.000456
9,primary_ai_tool,0.000065,0.002218
10,ai_tools_used_per_day,0.000055,0.000385


## 8. Verify agreement with the published experiment

Both entry points must produce the same non-timing metrics on the same data and dependency versions. Fit/prediction durations vary with hardware and system load.

In [8]:
published = json.loads((ROOT / 'reports' / 'experiment.json').read_text())
current = experiment['metadata']
assert current['dataset_sha256'] == published['dataset_sha256']
assert current['split_sha256'] == published['split_sha256']
assert current['selected_model'] == published['selected_model']
for a, b in zip(current['metrics'], published['metrics']):
    assert a['model'] == b['model']
    for metric in a:
        if metric != 'model' and 'seconds' not in metric:
            np.testing.assert_allclose(a[metric], b[metric], atol=1e-10, rtol=1e-10)
print('Dataset, split, selected model, and all non-timing metrics match the published experiment.')


Dataset, split, selected model, and all non-timing metrics match the published experiment.


## Conclusions and limitations

Linear Regression was selected by training CV and also achieved the strongest held-out metrics in this run. More complex trees did not improve this comparison.

The corrected evaluation does not resolve data provenance, possible concurrent/outcome-dependent predictors, or population validity. The original project had already inspected the dataset and test data, so this revised holdout is not a new untouched external validation. No temporal/employer-group validation or uncertainty intervals are claimed.

Before deployment claims: verify the dataset card, define prediction-time feature availability, perform a pre-specified sensitivity experiment without satisfaction/productivity, and evaluate on independent data.
